# QRAG: Comprehensive Generation Pipeline

**Research Questions:**
1. Does quantization degrade answer accuracy (RAG vs parametric knowledge)?
2. Does quantization affect faithfulness and hallucination rates?
3. Does quantization hurt long-context reasoning disproportionately?

**Generation Sets:**
- Set A: Answer Quality & Faithfulness (100 samples, stratified by question type)
- Set B: Context Length Robustness (240 samples, 4 lengths × 3 positions × 20 samples)
- Set C: Output Consistency (150 generations = 30 samples × 5 runs)
- Set D: Distractor Sensitivity (150 generations = 75 samples × 2 conditions)

**Models Evaluated:**
- FP16 (baseline): Mistral-7B-v0.1, Mistral-7B-Instruct-v0.1
- AWQ (4-bit): TheBloke variants
- NF4 (4-bit): BitsAndBytes quantization
- GPTQ (4-bit): TheBloke variants

Total per model: 740 generations
Total for 8 model configs: 5,920 generations

Setup and Dependencies

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch',
    'transformers==4.51.3',
    'accelerate',
    'bitsandbytes',
    'datasets',
    'autoawq',
    'exllamav2',
    'numpy',
    'scipy',
    'huggingface-hub'
], check=True)

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Dependencies installed")

In [ ]:
import gc
import time
import json
import torch
import random
import logging
import zipfile
import numpy as np
import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from datasets import load_dataset

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Imports complete")

Configuration

In [ ]:
# RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# RUN_ID = f"qrag_gen_{RUN_TIMESTAMP}"
RUN_ID = f"qrag_gen_20251228_115849"

OUTPUT_DIR = Path(f'/kaggle/working/{RUN_ID}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

MAX_NEW_TOKENS = 128
MAX_CONTEXT_TOKENS = 4096
TEMPERATURE = 0.7
TOP_K = 50
TOP_P = 0.9

SET_A_DISTRIBUTION = {
    'factual': 30,
    'multihop': 25,
    'numerical': 20,
    'unanswerable': 25
}

CONTEXT_LENGTHS = [512, 1024, 2048, 4096]
ANSWER_POSITIONS = ['start', 'middle', 'end']
SAMPLES_PER_CONFIG = 20

CONSISTENCY_SAMPLES = 30
CONSISTENCY_RUNS = 5

DISTRACTOR_SAMPLES = 75

TOTAL_SET_A = sum(SET_A_DISTRIBUTION.values()) * 2
TOTAL_SET_B = len(CONTEXT_LENGTHS) * len(ANSWER_POSITIONS) * SAMPLES_PER_CONFIG
TOTAL_SET_C = CONSISTENCY_SAMPLES * CONSISTENCY_RUNS
TOTAL_SET_D = DISTRACTOR_SAMPLES * 2

logger.info(f"Run ID: {RUN_ID}")
logger.info(f"Output: {OUTPUT_DIR}")
logger.info(f"Device: {DEVICE}")
logger.info(f"Random seed: {RANDOM_SEED}")
logger.info("")
logger.info("Generation plan per model:")
logger.info(f"  Set A (Answer Quality): {TOTAL_SET_A} generations (100 samples × 2 conditions)")
logger.info(f"  Set B (Context Robustness): {TOTAL_SET_B} generations (4 lengths × 3 positions × 20 samples)")
logger.info(f"  Set C (Output Consistency): {TOTAL_SET_C} generations (30 samples × 5 runs)")
logger.info(f"  Set D (Distractor Sensitivity): {TOTAL_SET_D} generations (75 samples × 2 conditions)")
logger.info(f"  Total per model: {TOTAL_SET_A + TOTAL_SET_B + TOTAL_SET_C + TOTAL_SET_D} generations")
logger.info("")
logger.info(f"Total for 8 model configs: {(TOTAL_SET_A + TOTAL_SET_B + TOTAL_SET_C + TOTAL_SET_D) * 8} generations")

if torch.cuda.is_available():
    logger.info("")
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Utility Functions

In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

def save_json(data: Dict, output_path: Path):
    def convert(obj):
        if isinstance(obj, (np.integer, np.int64)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, dict):
            return {key: convert(value) for key, value in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [convert(item) for item in obj]
        return obj
    
    converted = convert(data)
    with open(output_path, 'w') as f:
        json.dump(converted, f, indent=2)
    
    logger.info(f"Saved: {output_path.name} ({output_path.stat().st_size / 1024:.1f} KB)")

def count_tokens(tokenizer, text: str) -> int:
    encoding = tokenizer(text, return_tensors='pt', truncation=False, add_special_tokens=True)
    return encoding['input_ids'].shape[1]

def truncate_to_tokens(tokenizer, text: str, max_tokens: int) -> str:
    encoding = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_tokens, add_special_tokens=True)
    return tokenizer.decode(encoding['input_ids'][0], skip_special_tokens=True)

def get_model_device(model):
    if hasattr(model, 'device'):
        return model.device
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("Utility functions defined")

Dataset Loading

In [ ]:
def load_set_a_samples() -> List[Dict]:
    logger.info("Loading Set A samples")
    logger.info(f"Target distribution: {SET_A_DISTRIBUTION}")
    
    samples = []
    
    squad = load_dataset('squad_v2', split='validation')
    factual_samples = []
    for item in squad:
        if len(factual_samples) >= SET_A_DISTRIBUTION['factual']:
            break
        if item['answers']['text']:
            factual_samples.append({
                'question': item['question'].strip(),
                'context': item['context'].strip(),
                'answer': item['answers']['text'][0],
                'answer_variants': item['answers']['text'],
                'category': 'factual',
                'source': 'squad_v2'
            })
    samples.extend(factual_samples)
    logger.info(f"  Loaded {len(factual_samples)} factual samples")
    
    hotpot = load_dataset('hotpot_qa', 'distractor', split='validation')
    multihop_samples = []
    for item in hotpot:
        if len(multihop_samples) >= SET_A_DISTRIBUTION['multihop']:
            break
        if item['answer']:
            context = ' '.join(item['context']['sentences'][0])
            multihop_samples.append({
                'question': item['question'].strip(),
                'context': context[:2000],
                'answer': item['answer'],
                'answer_variants': [item['answer']],
                'category': 'multihop',
                'source': 'hotpot_qa'
            })
    samples.extend(multihop_samples)
    logger.info(f"  Loaded {len(multihop_samples)} multihop samples")
    
    drop = load_dataset('drop', split='validation')
    numerical_samples = []
    for item in drop:
        if len(numerical_samples) >= SET_A_DISTRIBUTION['numerical']:
            break
        if item['answers_spans']['spans']:
            numerical_samples.append({
                'question': item['question'].strip(),
                'context': item['passage'].strip(),
                'answer': item['answers_spans']['spans'][0],
                'answer_variants': item['answers_spans']['spans'],
                'category': 'numerical',
                'source': 'drop'
            })
    samples.extend(numerical_samples)
    logger.info(f"  Loaded {len(numerical_samples)} numerical samples")
    
    unanswerable_samples = []
    for item in squad:
        if len(unanswerable_samples) >= SET_A_DISTRIBUTION['unanswerable']:
            break
        if not item['answers']['text']:
            unanswerable_samples.append({
                'question': item['question'].strip(),
                'context': item['context'].strip(),
                'answer': "",
                'answer_variants': [],
                'category': 'unanswerable',
                'source': 'squad_v2'
            })
    samples.extend(unanswerable_samples)
    logger.info(f"  Loaded {len(unanswerable_samples)} unanswerable samples")
    
    random.Random(RANDOM_SEED).shuffle(samples)
    logger.info(f"Total Set A samples: {len(samples)}")
    return samples

def load_set_b_base_samples() -> List[Dict]:
    logger.info("Loading Set B base samples")
    needed = len(CONTEXT_LENGTHS) * len(ANSWER_POSITIONS) * SAMPLES_PER_CONFIG
    logger.info(f"Need {needed} samples")
    
    dataset = load_dataset('squad_v2', split='validation')
    samples = []
    
    for item in dataset:
        if len(samples) >= needed:
            break
        if item['answers']['text']:
            samples.append({
                'question': item['question'].strip(),
                'context': item['context'].strip(),
                'answer': item['answers']['text'][0],
                'answer_variants': item['answers']['text'],
                'source': 'squad_v2'
            })
    
    logger.info(f"Loaded {len(samples)} base samples")
    return samples

def load_distractor_corpus() -> List[str]:
    logger.info("Loading distractor corpus from WikiText-103")
    
    dataset = load_dataset('wikitext', 'wikitext-103-v1', split='train', streaming=True)
    distractors = []
    
    for item in dataset:
        text = item['text'].strip()
        if len(text) < 100 or text.startswith('=') or text.startswith(' ='):
            continue
        distractors.append(text)
        if len(distractors) >= 2000:
            break
    
    logger.info(f"Loaded {len(distractors)} distractor passages")
    return distractors

def load_set_d_samples() -> List[Dict]:
    logger.info("Loading Set D distractor sensitivity samples")
    
    dataset = load_dataset('squad_v2', split='validation')
    samples = []
    
    for item in dataset:
        if len(samples) >= DISTRACTOR_SAMPLES:
            break
        if item['answers']['text'] and 200 <= len(item['context']) <= 400:
            samples.append({
                'question': item['question'].strip(),
                'context': item['context'].strip(),
                'answer': item['answers']['text'][0],
                'answer_variants': item['answers']['text'],
                'source': 'squad_v2'
            })
    
    logger.info(f"Loaded {len(samples)} Set D samples")
    return samples

print("Dataset loading functions defined")

Context Manipulation

In [ ]:
class DistractorPool:
    def __init__(self, distractors: List[str], seed: int = 42):
        self.distractors = distractors
        self.rng = random.Random(seed)
    
    def get_distractor_text(self, target_chars: int) -> str:
        result = []
        current_length = 0
        available = self.distractors.copy()
        self.rng.shuffle(available)
        
        for passage in available:
            if current_length >= target_chars:
                break
            result.append(passage)
            current_length += len(passage) + 1
        
        return ' '.join(result)

def pad_context_to_length(
    tokenizer,
    original_context: str,
    answer: str,
    target_tokens: int,
    position: str,
    distractor_pool: DistractorPool
) -> Tuple[str, int]:
    current_tokens = count_tokens(tokenizer, original_context)
    
    if current_tokens >= target_tokens:
        return truncate_to_tokens(tokenizer, original_context, target_tokens), target_tokens
    
    tokens_needed = target_tokens - current_tokens
    chars_needed = tokens_needed * 4
    
    distractor_text = distractor_pool.get_distractor_text(chars_needed)
    distractor_truncated = truncate_to_tokens(tokenizer, distractor_text, tokens_needed)
    
    if position == 'start':
        padded = original_context + " " + distractor_truncated
    elif position == 'end':
        padded = distractor_truncated + " " + original_context
    else:
        half = tokens_needed // 2
        distractor_before = truncate_to_tokens(tokenizer, distractor_text, half)
        distractor_text_2 = distractor_pool.get_distractor_text(chars_needed)
        distractor_after = truncate_to_tokens(tokenizer, distractor_text_2, tokens_needed - half)
        padded = distractor_before + " " + original_context + " " + distractor_after
    
    actual_tokens = count_tokens(tokenizer, padded)
    
    if actual_tokens > target_tokens:
        padded = truncate_to_tokens(tokenizer, padded, target_tokens)
        actual_tokens = target_tokens
    
    return padded, actual_tokens

def add_distractors(
    original_context: str,
    distractor_pool: DistractorPool
) -> str:
    distractor_1 = distractor_pool.distractors[distractor_pool.rng.randint(0, len(distractor_pool.distractors)-1)][:200]
    distractor_2 = distractor_pool.distractors[distractor_pool.rng.randint(0, len(distractor_pool.distractors)-1)][:200]
    distractor_3 = distractor_pool.distractors[distractor_pool.rng.randint(0, len(distractor_pool.distractors)-1)][:200]
    
    half_point = len(original_context) // 2
    context_part1 = original_context[:half_point]
    context_part2 = original_context[half_point:]
    
    return f"{distractor_1} {context_part1} {distractor_2} {context_part2} {distractor_3}"

print("Context manipulation functions defined")

Prompt Templates

In [ ]:
def build_base_rag_prompt(context: str, question: str) -> str:
    return f"{context}\n\nQ: {question}\nA:"

def build_base_no_rag_prompt(question: str) -> str:
    return f"Q: {question}\nA:"

def build_instruct_rag_prompt(context: str, question: str) -> str:
    return f"[INST] {context}\n\n{question} [/INST]"

def build_instruct_no_rag_prompt(question: str) -> str:
    return f"[INST] {question} [/INST]"

def extract_answer(response: str, prompt: str) -> str:
    answer = response.strip()
    
    if prompt in answer:
        answer = answer.replace(prompt, '').strip()
    
    if '\n' in answer:
        answer = answer.split('\n')[0].strip()
    
    prefixes = ['Answer:', 'A:', 'answer:', 'a:']
    for prefix in prefixes:
        if answer.startswith(prefix):
            answer = answer[len(prefix):].strip()
    
    if len(answer.split()) > 50:
        sentences = answer.split('.')
        if sentences:
            answer = sentences[0] + '.'
    
    return answer

print("Prompt functions defined")

Model Loading

In [ ]:
def load_transformers_model(model_name: str, quantization: str):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    logger.info(f"Loading: {model_name}")
    logger.info(f"Quantization: {quantization}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    if quantization == 'fp16':
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map='auto',
            use_cache=True
        )
    elif quantization == 'nf4':
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map='auto',
            use_cache=True
        )
    
    model.eval()
    logger.info(f"Model loaded on {get_model_device(model)}")
    return model, tokenizer

def load_awq_model(model_name: str):
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer
    
    logger.info(f"Loading: {model_name}")
    logger.info(f"Quantization: AWQ")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoAWQForCausalLM.from_quantized(
        model_name,
        fuse_layers=True,
        use_cache=True
    )
    model.eval()
    
    logger.info(f"AWQ model loaded on {get_model_device(model)}")
    return model, tokenizer

def load_gptq_model(model_name: str):
    from exllamav2 import ExLlamaV2, ExLlamaV2Config, ExLlamaV2Cache, ExLlamaV2Tokenizer
    from huggingface_hub import snapshot_download
    
    logger.info(f"Loading: {model_name}")
    logger.info(f"Quantization: GPTQ")
    
    model_dir = snapshot_download(
        model_name,
        allow_patterns=["*.json", "*.safetensors", "*.model"]
    )
    
    config = ExLlamaV2Config()
    config.model_dir = model_dir
    config.prepare()
    config.max_seq_len = 4096
    
    model = ExLlamaV2(config)
    cache = ExLlamaV2Cache(model, lazy=True, max_seq_len=4096)
    model.load_autosplit(cache)
    
    tokenizer = ExLlamaV2Tokenizer(config)
    
    logger.info("GPTQ model loaded")
    return model, tokenizer, cache

def cleanup_model(model, tokenizer, cache=None):
    del model
    del tokenizer
    if cache is not None:
        del cache
    clear_memory()
    logger.info("Model cleaned up")

print("Model loading functions defined")

Generation Functions

In [ ]:
def generate_transformers(model, tokenizer, prompt: str, temperature: float, rep_penalty: float) -> str:
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    
    device = get_model_device(model)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096)
    input_length = inputs['input_ids'].shape[1]
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=temperature,
            top_k=TOP_K,
            top_p=TOP_P,
            repetition_penalty=rep_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

def generate_exllama(model, tokenizer, cache, prompt: str, temperature: float, rep_penalty: float) -> str:
    from exllamav2.generator import ExLlamaV2StreamingGenerator, ExLlamaV2Sampler
    
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    
    generator = ExLlamaV2StreamingGenerator(model, cache, tokenizer)
    
    settings = ExLlamaV2Sampler.Settings()
    settings.temperature = temperature
    settings.top_k = TOP_K
    settings.top_p = TOP_P
    settings.token_repetition_penalty = rep_penalty
    
    input_ids = tokenizer.encode(prompt)
    generator.begin_stream(input_ids, settings)
    
    output_tokens = []
    for _ in range(MAX_NEW_TOKENS):
        chunk, eos, _ = generator.stream()
        output_tokens.append(chunk)
        if eos:
            break
    
    return ''.join(output_tokens).strip()

print("Generation functions defined")

Set Generation Functions

In [ ]:
def generate_set_a(model, tokenizer, samples: List[Dict], variant: str, is_gptq: bool = False, cache=None) -> List[Dict]:
    logger.info(f"Generating Set A: Answer Quality ({len(samples)} samples)")
    logger.info(f"  Total generations: {len(samples) * 2} (RAG + No-RAG)")
    
    if variant == 'base':
        build_rag = build_base_rag_prompt
        build_no_rag = build_base_no_rag_prompt
        temp = 0.3
        rep_penalty = 1.3
    else:
        build_rag = build_instruct_rag_prompt
        build_no_rag = build_instruct_no_rag_prompt
        temp = TEMPERATURE
        rep_penalty = 1.1
    
    generate_fn = lambda p: generate_exllama(model, tokenizer, cache, p, temp, rep_penalty) if is_gptq else generate_transformers(model, tokenizer, p, temp, rep_penalty)
    
    results = []
    category_counts = defaultdict(int)
    set_start = time.time()
    
    for i, sample in enumerate(samples):
        sample_start = time.time()
        
        context = sample['context']
        if count_tokens(tokenizer, context) > MAX_CONTEXT_TOKENS:
            context = truncate_to_tokens(tokenizer, context, MAX_CONTEXT_TOKENS)
        
        rag_prompt = build_rag(context, sample['question'])
        rag_start = time.perf_counter()
        rag_response = generate_fn(rag_prompt)
        rag_time = (time.perf_counter() - rag_start) * 1000
        rag_prediction = extract_answer(rag_response, rag_prompt)
        
        no_rag_prompt = build_no_rag(sample['question'])
        no_rag_start = time.perf_counter()
        no_rag_response = generate_fn(no_rag_prompt)
        no_rag_time = (time.perf_counter() - no_rag_start) * 1000
        no_rag_prediction = extract_answer(no_rag_response, no_rag_prompt)
        
        results.append({
            'sample_id': i,
            'question': sample['question'],
            'context': context,
            'context_length_tokens': count_tokens(tokenizer, context),
            'ground_truth': sample['answer'],
            'ground_truth_variants': sample['answer_variants'],
            'category': sample['category'],
            'source': sample['source'],
            'rag_prediction': rag_prediction,
            'rag_generation_time_ms': rag_time,
            'no_rag_prediction': no_rag_prediction,
            'no_rag_generation_time_ms': no_rag_time
        })
        
        category_counts[sample['category']] += 1
        sample_time = time.time() - sample_start
        
        if (i + 1) % 10 == 0 or (i + 1) == len(samples):
            elapsed = time.time() - set_start
            avg_time = elapsed / (i + 1)
            remaining = (len(samples) - (i + 1)) * avg_time
            logger.info(f"    Sample {i + 1}/{len(samples)} | Category: {sample['category']} | "
                       f"Time: {sample_time:.1f}s | Elapsed: {elapsed/60:.1f}m | ETA: {remaining/60:.1f}m")
    
    logger.info(f"  Set A complete: {len(results)} samples in {(time.time() - set_start)/60:.1f} minutes")
    logger.info(f"  By category: {dict(category_counts)}")
    return results

def generate_set_b(model, tokenizer, base_samples: List[Dict], variant: str, distractor_pool: DistractorPool, is_gptq: bool = False, cache=None) -> List[Dict]:
    logger.info(f"Generating Set B: Context Length Robustness")
    total_samples = len(CONTEXT_LENGTHS) * len(ANSWER_POSITIONS) * SAMPLES_PER_CONFIG
    logger.info(f"  Total generations: {total_samples}")
    
    if variant == 'base':
        build_rag = build_base_rag_prompt
        temp = 0.3
        rep_penalty = 1.3
    else:
        build_rag = build_instruct_rag_prompt
        temp = TEMPERATURE
        rep_penalty = 1.1
    
    generate_fn = lambda p: generate_exllama(model, tokenizer, cache, p, temp, rep_penalty) if is_gptq else generate_transformers(model, tokenizer, p, temp, rep_penalty)
    
    results = []
    sample_idx = 0
    set_start = time.time()
    config_num = 0
    total_configs = len(CONTEXT_LENGTHS) * len(ANSWER_POSITIONS)
    
    for length in CONTEXT_LENGTHS:
        for position in ANSWER_POSITIONS:
            config_num += 1
            config_start = time.time()
            logger.info(f"  Config {config_num}/{total_configs}: length={length}, position={position}")
            
            for i in range(SAMPLES_PER_CONFIG):
                if sample_idx >= len(base_samples):
                    break
                
                sample = base_samples[sample_idx]
                sample_idx += 1
                
                padded_context, actual_tokens = pad_context_to_length(
                    tokenizer,
                    sample['context'],
                    sample['answer'],
                    length,
                    position,
                    distractor_pool
                )
                
                rag_prompt = build_rag(padded_context, sample['question'])
                rag_start = time.perf_counter()
                rag_response = generate_fn(rag_prompt)
                rag_time = (time.perf_counter() - rag_start) * 1000
                rag_prediction = extract_answer(rag_response, rag_prompt)
                
                results.append({
                    'sample_id': sample_idx - 1,
                    'question': sample['question'],
                    'original_context': sample['context'],
                    'padded_context': padded_context,
                    'target_length_tokens': length,
                    'actual_length_tokens': actual_tokens,
                    'answer_position': position,
                    'ground_truth': sample['answer'],
                    'ground_truth_variants': sample['answer_variants'],
                    'source': sample['source'],
                    'rag_prediction': rag_prediction,
                    'rag_generation_time_ms': rag_time
                })
                
                if (i + 1) % 5 == 0 or (i + 1) == SAMPLES_PER_CONFIG:
                    config_elapsed = time.time() - config_start
                    config_avg = config_elapsed / (i + 1)
                    config_remaining = (SAMPLES_PER_CONFIG - (i + 1)) * config_avg
                    logger.info(f"      Sample {i + 1}/{SAMPLES_PER_CONFIG} | "
                               f"Elapsed: {config_elapsed/60:.1f}m | ETA: {config_remaining/60:.1f}m")
            
            config_time = time.time() - config_start
            logger.info(f"    Config complete in {config_time/60:.1f} minutes")
    
    logger.info(f"  Set B complete: {len(results)} samples in {(time.time() - set_start)/60:.1f} minutes")
    return results

def generate_set_c(model, tokenizer, samples: List[Dict], variant: str, is_gptq: bool = False, cache=None) -> List[Dict]:
    logger.info(f"Generating Set C: Output Consistency ({len(samples)} samples × {CONSISTENCY_RUNS} runs)")
    logger.info(f"  Total generations: {len(samples) * CONSISTENCY_RUNS}")
    
    if variant == 'base':
        build_rag = build_base_rag_prompt
        temp = 0.3
        rep_penalty = 1.3
    else:
        build_rag = build_instruct_rag_prompt
        temp = TEMPERATURE
        rep_penalty = 1.1
    
    results = []
    set_start = time.time()
    
    for i, sample in enumerate(samples):
        sample_start = time.time()
        
        context = sample['context']
        if count_tokens(tokenizer, context) > MAX_CONTEXT_TOKENS:
            context = truncate_to_tokens(tokenizer, context, MAX_CONTEXT_TOKENS)
        
        rag_prompt = build_rag(context, sample['question'])
        
        for run_id in range(CONSISTENCY_RUNS):
            torch.manual_seed(RANDOM_SEED + run_id)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(RANDOM_SEED + run_id)
            
            if is_gptq:
                rag_response = generate_exllama(model, tokenizer, cache, rag_prompt, temp, rep_penalty)
            else:
                rag_response = generate_transformers(model, tokenizer, rag_prompt, temp, rep_penalty)
            
            rag_prediction = extract_answer(rag_response, rag_prompt)
            
            results.append({
                'sample_id': i,
                'run_id': run_id,
                'question': sample['question'],
                'context': context,
                'ground_truth': sample['answer'],
                'ground_truth_variants': sample['answer_variants'],
                'category': sample['category'],
                'source': sample['source'],
                'rag_prediction': rag_prediction
            })
        
        sample_time = time.time() - sample_start
        
        if (i + 1) % 5 == 0 or (i + 1) == len(samples):
            elapsed = time.time() - set_start
            avg_time = elapsed / (i + 1)
            remaining = (len(samples) - (i + 1)) * avg_time
            total_gens = (i + 1) * CONSISTENCY_RUNS
            logger.info(f"    Sample {i + 1}/{len(samples)} ({total_gens} generations) | "
                       f"Time: {sample_time:.1f}s | Elapsed: {elapsed/60:.1f}m | ETA: {remaining/60:.1f}m")
    
    logger.info(f"  Set C complete: {len(results)} generations in {(time.time() - set_start)/60:.1f} minutes")
    return results

def generate_set_d(model, tokenizer, samples: List[Dict], variant: str, distractor_pool: DistractorPool, is_gptq: bool = False, cache=None) -> List[Dict]:
    logger.info(f"Generating Set D: Distractor Sensitivity ({len(samples)} samples × 2 conditions)")
    logger.info(f"  Total generations: {len(samples) * 2}")
    
    if variant == 'base':
        build_rag = build_base_rag_prompt
        temp = 0.3
        rep_penalty = 1.3
    else:
        build_rag = build_instruct_rag_prompt
        temp = TEMPERATURE
        rep_penalty = 1.1
    
    generate_fn = lambda p: generate_exllama(model, tokenizer, cache, p, temp, rep_penalty) if is_gptq else generate_transformers(model, tokenizer, p, temp, rep_penalty)
    
    results = []
    set_start = time.time()
    
    for i, sample in enumerate(samples):
        sample_start = time.time()
        
        clean_prompt = build_rag(sample['context'], sample['question'])
        clean_start = time.perf_counter()
        clean_response = generate_fn(clean_prompt)
        clean_time = (time.perf_counter() - clean_start) * 1000
        clean_prediction = extract_answer(clean_response, clean_prompt)
        
        distracted_context = add_distractors(sample['context'], distractor_pool)
        distracted_prompt = build_rag(distracted_context, sample['question'])
        distracted_start = time.perf_counter()
        distracted_response = generate_fn(distracted_prompt)
        distracted_time = (time.perf_counter() - distracted_start) * 1000
        distracted_prediction = extract_answer(distracted_response, distracted_prompt)
        
        results.append({
            'sample_id': i,
            'question': sample['question'],
            'clean_context': sample['context'],
            'distracted_context': distracted_context,
            'ground_truth': sample['answer'],
            'ground_truth_variants': sample['answer_variants'],
            'source': sample['source'],
            'clean_prediction': clean_prediction,
            'clean_generation_time_ms': clean_time,
            'distracted_prediction': distracted_prediction,
            'distracted_generation_time_ms': distracted_time
        })
        
        sample_time = time.time() - sample_start
        
        if (i + 1) % 15 == 0 or (i + 1) == len(samples):
            elapsed = time.time() - set_start
            avg_time = elapsed / (i + 1)
            remaining = (len(samples) - (i + 1)) * avg_time
            total_gens = (i + 1) * 2
            logger.info(f"    Sample {i + 1}/{len(samples)} ({total_gens} generations) | "
                       f"Time: {sample_time:.1f}s | Elapsed: {elapsed/60:.1f}m | ETA: {remaining/60:.1f}m")
    
    logger.info(f"  Set D complete: {len(results)} samples in {(time.time() - set_start)/60:.1f} minutes")
    return results

print("Set generation functions defined")

Save Results Functions

In [ ]:
def save_results(config_name: str, set_name: str, results: List[Dict], metadata: Dict):
    complete_output = {
        'model_config': metadata['model_config'],
        'set_name': set_name,
        'samples': results,
        'count': len(results),
        'generation_metadata': metadata['generation_metadata']
    }
    
    complete_file = OUTPUT_DIR / f'{config_name}_{set_name}_complete.json'
    save_json(complete_output, complete_file)
    
    categories = {}
    for sample in results:
        if 'category' in sample:
            cat = sample['category']
        elif 'target_length_tokens' in sample and 'answer_position' in sample:
            cat = f"{sample['target_length_tokens']}_{sample['answer_position']}"
        elif 'run_id' in sample:
            cat = sample['sample_id']
        elif 'clean_prediction' in sample:
            cat = 'distractor'
        else:
            cat = 'other'
        
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(sample)
    
    minimal_samples = []
    for cat, cat_samples in categories.items():
        minimal_samples.append(cat_samples[0])
    
    minimal_output = {
        'model_config': metadata['model_config'],
        'set_name': set_name,
        'samples': minimal_samples,
        'count': len(minimal_samples),
        'generation_metadata': metadata['generation_metadata']
    }
    
    minimal_file = OUTPUT_DIR / f'{config_name}_{set_name}_minimal.json'
    save_json(minimal_output, minimal_file)

print("Save results functions defined")

Load Datasets

In [ ]:
logger.info("")
logger.info("LOADING DATASETS")

set_a_samples = load_set_a_samples()
set_b_base_samples = load_set_b_base_samples()
distractor_corpus = load_distractor_corpus()
distractor_pool = DistractorPool(distractor_corpus, seed=RANDOM_SEED)

set_c_samples = []
for category in SET_A_DISTRIBUTION.keys():
    cat_samples = [s for s in set_a_samples if s['category'] == category]
    samples_to_take = min(10, len(cat_samples))
    set_c_samples.extend(cat_samples[:samples_to_take])

set_d_samples = load_set_d_samples()

logger.info("")
logger.info("Dataset loading complete")
logger.info(f"  Set A samples: {len(set_a_samples)}")
logger.info(f"  Set B base samples: {len(set_b_base_samples)}")
logger.info(f"  Set C samples: {len(set_c_samples)}")
logger.info(f"  Set D samples: {len(set_d_samples)}")
logger.info(f"  Distractor passages: {len(distractor_corpus)}")

FP16 Base Model

In [ ]:
# logger.info("")
# logger.info("CONFIGURATION: FP16_BASE")

# config_start = time.time()
# model, tokenizer = load_transformers_model('mistralai/Mistral-7B-v0.1', 'fp16')

# metadata = {
#     'model_config': {
#         'name': 'mistralai/Mistral-7B-v0.1',
#         'quantization': 'fp16',
#         'variant': 'base'
#     },
#     'generation_metadata': {
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
#         'run_id': RUN_ID,
#         'max_new_tokens': MAX_NEW_TOKENS,
#         'temperature': TEMPERATURE,
#         'top_k': TOP_K,
#         'top_p': TOP_P,
#         'random_seed': RANDOM_SEED
#     }
# }

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'base')
# save_results('fp16_base', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'base', distractor_pool)
# save_results('fp16_base', 'set_b', set_b_results, metadata)

# logger.info("Generating Set C")
# set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'base')
# save_results('fp16_base', 'set_c', set_c_results, metadata)

# logger.info("Generating Set D")
# set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'base', distractor_pool)
# save_results('fp16_base', 'set_d', set_d_results, metadata)

# config_time = time.time() - config_start
# logger.info(f"FP16_BASE completed in {config_time/60:.1f} minutes")

# cleanup_model(model, tokenizer)
# time.sleep(2)

FP16 Instruct Model

In [ ]:
# logger.info("")
# logger.info("CONFIGURATION: FP16_INSTRUCT")

# config_start = time.time()
# model, tokenizer = load_transformers_model('mistralai/Mistral-7B-Instruct-v0.1', 'fp16')

# metadata = {
#     'model_config': {
#         'name': 'mistralai/Mistral-7B-Instruct-v0.1',
#         'quantization': 'fp16',
#         'variant': 'instruct'
#     },
#     'generation_metadata': {
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
#         'run_id': RUN_ID,
#         'max_new_tokens': MAX_NEW_TOKENS,
#         'temperature': TEMPERATURE,
#         'top_k': TOP_K,
#         'top_p': TOP_P,
#         'random_seed': RANDOM_SEED
#     }
# }

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'instruct')
# save_results('fp16_instruct', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'instruct', distractor_pool)
# save_results('fp16_instruct', 'set_b', set_b_results, metadata)

# logger.info("Generating Set C")
# set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'instruct')
# save_results('fp16_instruct', 'set_c', set_c_results, metadata)

# logger.info("Generating Set D")
# set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'instruct', distractor_pool)
# save_results('fp16_instruct', 'set_d', set_d_results, metadata)

# config_time = time.time() - config_start
# logger.info(f"FP16_INSTRUCT completed in {config_time/60:.1f} minutes")

# cleanup_model(model, tokenizer)
# time.sleep(2)

AWQ Base Model

In [ ]:
# logger.info("")
# logger.info("CONFIGURATION: AWQ_BASE")

# config_start = time.time()
# model, tokenizer = load_awq_model('TheBloke/Mistral-7B-v0.1-AWQ')

# metadata = {
#     'model_config': {
#         'name': 'TheBloke/Mistral-7B-v0.1-AWQ',
#         'quantization': 'awq',
#         'variant': 'base'
#     },
#     'generation_metadata': {
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
#         'run_id': RUN_ID,
#         'max_new_tokens': MAX_NEW_TOKENS,
#         'temperature': TEMPERATURE,
#         'top_k': TOP_K,
#         'top_p': TOP_P,
#         'random_seed': RANDOM_SEED
#     }
# }

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'base')
# save_results('awq_base', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'base', distractor_pool)
# save_results('awq_base', 'set_b', set_b_results, metadata)

# logger.info("Generating Set C")
# set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'base')
# save_results('awq_base', 'set_c', set_c_results, metadata)

# logger.info("Generating Set D")
# set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'base', distractor_pool)
# save_results('awq_base', 'set_d', set_d_results, metadata)

# config_time = time.time() - config_start
# logger.info(f"AWQ_BASE completed in {config_time/60:.1f} minutes")

# cleanup_model(model, tokenizer)
# time.sleep(2)

AWQ Instruct Model

In [ ]:
# logger.info("")
# logger.info("CONFIGURATION: AWQ_INSTRUCT")

# config_start = time.time()
# model, tokenizer = load_awq_model('TheBloke/Mistral-7B-Instruct-v0.1-AWQ')

# metadata = {
#     'model_config': {
#         'name': 'TheBloke/Mistral-7B-Instruct-v0.1-AWQ',
#         'quantization': 'awq',
#         'variant': 'instruct'
#     },
#     'generation_metadata': {
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
#         'run_id': RUN_ID,
#         'max_new_tokens': MAX_NEW_TOKENS,
#         'temperature': TEMPERATURE,
#         'top_k': TOP_K,
#         'top_p': TOP_P,
#         'random_seed': RANDOM_SEED
#     }
# }

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'instruct')
# save_results('awq_instruct', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'instruct', distractor_pool)
# save_results('awq_instruct', 'set_b', set_b_results, metadata)

# logger.info("Generating Set C")
# set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'instruct')
# save_results('awq_instruct', 'set_c', set_c_results, metadata)

# logger.info("Generating Set D")
# set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'instruct', distractor_pool)
# save_results('awq_instruct', 'set_d', set_d_results, metadata)

# config_time = time.time() - config_start
# logger.info(f"AWQ_INSTRUCT completed in {config_time/60:.1f} minutes")

# cleanup_model(model, tokenizer)
# time.sleep(2)

NF4 Base Model

In [ ]:
logger.info("")
logger.info("CONFIGURATION: NF4_BASE")

config_start = time.time()
model, tokenizer = load_transformers_model('mistralai/Mistral-7B-v0.1', 'nf4')

metadata = {
    'model_config': {
        'name': 'mistralai/Mistral-7B-v0.1',
        'quantization': 'nf4',
        'variant': 'base'
    },
    'generation_metadata': {
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'run_id': RUN_ID,
        'max_new_tokens': MAX_NEW_TOKENS,
        'temperature': TEMPERATURE,
        'top_k': TOP_K,
        'top_p': TOP_P,
        'random_seed': RANDOM_SEED
    }
}

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'base')
# save_results('nf4_base', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'base', distractor_pool)
# save_results('nf4_base', 'set_b', set_b_results, metadata)

logger.info("Generating Set C")
set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'base')
save_results('nf4_base', 'set_c', set_c_results, metadata)

logger.info("Generating Set D")
set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'base', distractor_pool)
save_results('nf4_base', 'set_d', set_d_results, metadata)

config_time = time.time() - config_start
logger.info(f"NF4_BASE completed in {config_time/60:.1f} minutes")

cleanup_model(model, tokenizer)
time.sleep(2)

NF4 Instruct Model

In [ ]:
logger.info("")
logger.info("CONFIGURATION: NF4_INSTRUCT")

config_start = time.time()
model, tokenizer = load_transformers_model('mistralai/Mistral-7B-Instruct-v0.1', 'nf4')

metadata = {
    'model_config': {
        'name': 'mistralai/Mistral-7B-Instruct-v0.1',
        'quantization': 'nf4',
        'variant': 'instruct'
    },
    'generation_metadata': {
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'run_id': RUN_ID,
        'max_new_tokens': MAX_NEW_TOKENS,
        'temperature': TEMPERATURE,
        'top_k': TOP_K,
        'top_p': TOP_P,
        'random_seed': RANDOM_SEED
    }
}

logger.info("Generating Set A")
set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'instruct')
save_results('nf4_instruct', 'set_a', set_a_results, metadata)

logger.info("Generating Set B")
set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'instruct', distractor_pool)
save_results('nf4_instruct', 'set_b', set_b_results, metadata)

logger.info("Generating Set C")
set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'instruct')
save_results('nf4_instruct', 'set_c', set_c_results, metadata)

logger.info("Generating Set D")
set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'instruct', distractor_pool)
save_results('nf4_instruct', 'set_d', set_d_results, metadata)

config_time = time.time() - config_start
logger.info(f"NF4_INSTRUCT completed in {config_time/60:.1f} minutes")

cleanup_model(model, tokenizer)
time.sleep(2)

GPTQ Base Model

In [ ]:
# logger.info("")
# logger.info("CONFIGURATION: GPTQ_BASE")

# config_start = time.time()
# model, tokenizer, cache = load_gptq_model('TheBloke/Mistral-7B-v0.1-GPTQ')

# metadata = {
#     'model_config': {
#         'name': 'TheBloke/Mistral-7B-v0.1-GPTQ',
#         'quantization': 'gptq',
#         'variant': 'base'
#     },
#     'generation_metadata': {
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
#         'run_id': RUN_ID,
#         'max_new_tokens': MAX_NEW_TOKENS,
#         'temperature': TEMPERATURE,
#         'top_k': TOP_K,
#         'top_p': TOP_P,
#         'random_seed': RANDOM_SEED
#     }
# }

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'base', is_gptq=True, cache=cache)
# save_results('gptq_base', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'base', distractor_pool, is_gptq=True, cache=cache)
# save_results('gptq_base', 'set_b', set_b_results, metadata)

# logger.info("Generating Set C")
# set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'base', is_gptq=True, cache=cache)
# save_results('gptq_base', 'set_c', set_c_results, metadata)

# logger.info("Generating Set D")
# set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'base', distractor_pool, is_gptq=True, cache=cache)
# save_results('gptq_base', 'set_d', set_d_results, metadata)

# config_time = time.time() - config_start
# logger.info(f"GPTQ_BASE completed in {config_time/60:.1f} minutes")

# cleanup_model(model, tokenizer, cache)
# time.sleep(2)

GPTQ Instruct Model

In [ ]:
# logger.info("")
# logger.info("CONFIGURATION: GPTQ_INSTRUCT")

# config_start = time.time()
# model, tokenizer, cache = load_gptq_model('TheBloke/Mistral-7B-Instruct-v0.1-GPTQ')

# metadata = {
#     'model_config': {
#         'name': 'TheBloke/Mistral-7B-Instruct-v0.1-GPTQ',
#         'quantization': 'gptq',
#         'variant': 'instruct'
#     },
#     'generation_metadata': {
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
#         'run_id': RUN_ID,
#         'max_new_tokens': MAX_NEW_TOKENS,
#         'temperature': TEMPERATURE,
#         'top_k': TOP_K,
#         'top_p': TOP_P,
#         'random_seed': RANDOM_SEED
#     }
# }

# logger.info("Generating Set A")
# set_a_results = generate_set_a(model, tokenizer, set_a_samples, 'instruct', is_gptq=True, cache=cache)
# save_results('gptq_instruct', 'set_a', set_a_results, metadata)

# logger.info("Generating Set B")
# set_b_results = generate_set_b(model, tokenizer, set_b_base_samples, 'instruct', distractor_pool, is_gptq=True, cache=cache)
# save_results('gptq_instruct', 'set_b', set_b_results, metadata)

# logger.info("Generating Set C")
# set_c_results = generate_set_c(model, tokenizer, set_c_samples, 'instruct', is_gptq=True, cache=cache)
# save_results('gptq_instruct', 'set_c', set_c_results, metadata)

# logger.info("Generating Set D")
# set_d_results = generate_set_d(model, tokenizer, set_d_samples, 'instruct', distractor_pool, is_gptq=True, cache=cache)
# save_results('gptq_instruct', 'set_d', set_d_results, metadata)

# config_time = time.time() - config_start
# logger.info(f"GPTQ_INSTRUCT completed in {config_time/60:.1f} minutes")

# cleanup_model(model, tokenizer, cache)
# time.sleep(2)

Create ZIP Archive

In [ ]:
logger.info("")
logger.info("Creating ZIP archive of all results")

zip_path = OUTPUT_DIR / f'{RUN_ID}_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for json_file in OUTPUT_DIR.glob('*.json'):
        zipf.write(json_file, json_file.name)
        logger.info(f"  Added: {json_file.name}")

zip_size_mb = zip_path.stat().st_size / 1024**2
logger.info(f"")
logger.info(f"ZIP archive created: {zip_path.name}")
logger.info(f"Archive size: {zip_size_mb:.2f} MB")

output_files = list(OUTPUT_DIR.glob('*.json'))
logger.info(f"")
logger.info(f"Total JSON files: {len(output_files)}")
logger.info(f"Expected: {8 * 4 * 2} files (8 configs × 4 sets × 2 versions)")
logger.info(f"")
logger.info("GENERATION PIPELINE COMPLETE")
logger.info(f"Download: {zip_path}")